# Week 14: Gaussian Mixture Models

## Setup and Data Loading

Importing libraries needed for Gaussian Mixture Models, including tools for feature scaling and evaluating cluster quality, and loading the diabetes dataset used throughout this capstone.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

df = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')

X = df.drop('Diabetes_binary', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(70692, 21)


## Creating a Sample for GMM

GMM is more computationally expensive than k-means, since it must estimate a full covariance matrix for each cluster rather than a single center point. Using the same 1,000-row random sample approach from Week 11's DBSCAN and HAC analysis keeps this comparable to that prior work and keeps runtime reasonable.

In [2]:
np.random.seed(42)
sample_indices = np.random.choice(X_scaled.shape[0], size=1000, replace=False)
X_sample = X_scaled[sample_indices]
df_sample = df.iloc[sample_indices].copy()

print(X_sample.shape)

(1000, 21)


## Fitting a GMM with 2 Components

Starting with 2 components (GMM's term for clusters), since k-means, DBSCAN, and HAC from prior weeks all found 2 clusters to be meaningful for this dataset. Testing whether GMM's ellipsoidal clusters agree with this same structure.

In [3]:
gmm = GaussianMixture(n_components=2, random_state=42)
gmm_labels = gmm.fit_predict(X_sample)

df_sample['GMM_Cluster'] = gmm_labels

print(df_sample['GMM_Cluster'].value_counts())
print()
print(df_sample.groupby('GMM_Cluster')[['BMI', 'Age', 'GenHlth', 'HighBP', 'Diabetes_binary']].mean())

GMM_Cluster
1    680
0    320
Name: count, dtype: int64

                   BMI       Age   GenHlth    HighBP  Diabetes_binary
GMM_Cluster                                                          
0            31.456250  9.606250  3.668750  0.740625         0.700000
1            28.316176  8.327941  2.463235  0.470588         0.401471


## Checking Cluster Shape: Spherical or Ellipsoidal?

Examining whether GMM's clusters actually took on non-spherical (ellipsoidal) shapes on this dataset, or whether they ended up close to spherical, which would suggest GMM's extra flexibility over k-means didn't meaningfully change the result here.

In [4]:
for i, cov in enumerate(gmm.covariances_):
    eigenvalues = np.linalg.eigvalsh(cov)
    print(f"Cluster {i} covariance eigenvalue range: {eigenvalues.min():.3f} to {eigenvalues.max():.3f}")
    print(f"Ratio (max/min): {eigenvalues.max() / eigenvalues.min():.2f}")
    print()

Cluster 0 covariance eigenvalue range: 0.290 to 3.927
Ratio (max/min): 13.55

Cluster 1 covariance eigenvalue range: 0.000 to 2.100
Ratio (max/min): 2099669.81



## Addressing Covariance Instability with a Constrained Model

Cluster 1's covariance matrix showed signs of near-singularity (eigenvalue ratio over 2 million), likely due to estimating a full 21x21 covariance matrix from only 680 points. Testing GMM with a 'diagonal' covariance type, which estimates far fewer parameters per cluster, to see whether this produces more stable, reliable results.

In [5]:
gmm_diag = GaussianMixture(n_components=2, covariance_type='diag', random_state=42)
gmm_diag_labels = gmm_diag.fit_predict(X_sample)

df_sample['GMM_Diag_Cluster'] = gmm_diag_labels

print(df_sample['GMM_Diag_Cluster'].value_counts())
print()
print(df_sample.groupby('GMM_Diag_Cluster')[['BMI', 'Age', 'GenHlth', 'HighBP', 'Diabetes_binary']].mean())
print()

for i, cov in enumerate(gmm_diag.covariances_):
    print(f"Cluster {i} variance range: {cov.min():.3f} to {cov.max():.3f}, ratio: {cov.max()/cov.min():.2f}")

GMM_Diag_Cluster
1    680
0    320
Name: count, dtype: int64

                        BMI       Age   GenHlth    HighBP  Diabetes_binary
GMM_Diag_Cluster                                                          
0                 31.456250  9.606250  3.668750  0.740625         0.700000
1                 28.316176  8.327941  2.463235  0.470588         0.401471

Cluster 0 variance range: 0.636 to 3.212, ratio: 5.05
Cluster 1 variance range: 0.000 to 1.291, ratio: 1291036.33


## Identifying the Near-Zero Variance Feature

The instability persisted even with diagonal covariance, suggesting a specific feature has almost no variance within Cluster 1, rather than a general overfitting problem. Checking each feature's variance within Cluster 1 to find the culprit.

In [6]:
cluster1_data = X_sample[gmm_diag_labels == 1]
feature_variances = pd.Series(cluster1_data.var(axis=0), index=X.columns).sort_values()

print(feature_variances.head(5))

AnyHealthcare    4.437343e-31
Stroke           2.961310e-30
DiffWalk         1.687423e-29
PhysHlth         3.860054e-01
MentHlth         5.631961e-01
dtype: float64


## Finding: Near-Zero Variance Feature Caused Covariance Instability

Investigating the extreme eigenvalue ratio in Cluster 1 (over 2 million) revealed the root cause: the AnyHealthcare feature has essentially zero variance within this cluster (variance = 4.4e-31), meaning nearly every point in Cluster 1 shares the same value for this feature. Stroke and DiffWalk showed similarly near-zero variance. This makes sense given Cluster 1 represents the lower-risk group (lower BMI, younger, better general health): within a healthier subpopulation, near-universal healthcare coverage and the near-total absence of stroke history or difficulty walking leaves the model almost no variation to estimate for these features, causing the covariance matrix to become nearly singular regardless of covariance type (full or diagonal). This is a meaningful limitation of GMM to note for Milestone Three: unlike k-means, which only needs to estimate a single center point per cluster, GMM's covariance estimation can become numerically unstable when a cluster is unexpectedly homogeneous on certain features, a tradeoff for its added flexibility that k-means does not share.

## GMM vs. K-Means on the Same Sample

Running k-means with k=2 on this same 1,000-point sample to directly compare cluster assignments against GMM, checking how often the two methods agree on which cluster a given point belongs to.

In [7]:
kmeans_sample = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans_sample.fit_predict(X_sample)

agreement = (kmeans_labels == gmm_labels).mean()
agreement_flipped = (kmeans_labels == (1 - gmm_labels)).mean()
true_agreement = max(agreement, agreement_flipped)

print(f"K-Means and GMM agreement rate: {true_agreement * 100:.1f}%")

K-Means and GMM agreement rate: 87.4%


## Overall Summary: GMM on This Dataset

GMM with 2 components found a cluster structure highly consistent with k-means (87.4% agreement, nearly identical split percentages and diabetes rates), confirming that the two-cluster structure discovered across multiple methods this capstone (k-means, HAC, and now GMM) is a robust finding, not an artifact of any single algorithm's assumptions. GMM's added flexibility (ellipsoidal, non-spherical clusters) did make a real difference for a subset of points (the 12.6% where GMM and k-means disagreed), and Cluster 0 showed a genuinely stretched ellipsoid shape (eigenvalue ratio of 13.55) supporting this. However, this flexibility came with a real cost: Cluster 1's covariance estimation became unstable due to near-zero variance in the AnyHealthcare, Stroke, and DiffWalk features, a limitation k-means does not share, since it only estimates a single center point per cluster rather than a full covariance structure. This is a useful, balanced conclusion for Milestone Three: GMM offers real modeling flexibility over k-means, but that flexibility requires enough data and enough within-cluster variance per feature to estimate reliably, which is not guaranteed even in a reasonably sized dataset.